In [29]:
import polars as pl
from pathlib import Path

DATA_DIR = Path("./data/csiro-biomass")

train = pl.read_csv(DATA_DIR / "train.csv")
test = pl.read_csv(DATA_DIR / "test.csv")
train

FileNotFoundError: No such file or directory (os error 2): data/csiro-biomass/train.csv

In [2]:
train_cleaned = train.select("image_path", "target_name", "target").pivot(index="image_path", on="target_name", values="target")
test_df = test.with_columns(pl.lit(0).alias("target"))
test_cleaned = test_df.select("image_path", "target_name", "target").pivot(index="image_path", on="target_name", values="target")
train_cleaned.head(10)

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,f64,f64,f64,f64,f64
"""train/ID1011485656.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1012260530.jpg""",0.0,0.0,7.6,7.6,7.6
"""train/ID1025234388.jpg""",6.05,0.0,0.0,6.05,6.05
"""train/ID1028611175.jpg""",0.0,30.9703,24.2376,55.2079,24.2376
"""train/ID1035947949.jpg""",0.4343,23.2239,10.5261,34.1844,10.9605
"""train/ID1036339023.jpg""",23.0755,2.6135,32.191,57.88,55.2665
"""train/ID1049634115.jpg""",1.5083,3.0167,13.575,18.1,15.0833
"""train/ID1051144034.jpg""",55.32,0.0,0.0,55.32,55.32
"""train/ID1052620238.jpg""",0.0,11.2291,20.1707,31.3998,20.1707


In [3]:
test_cleaned

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,i32,i32,i32,i32,i32
"""test/ID1001187975.jpg""",0,0,0,0,0


In [9]:
train_cleaned[0]['image_path'][0]

'train/ID1011485656.jpg'

In [ ]:
import cv2
import albumentations as A
from tqdm import tqdm

augmented_images_dir = DATA_DIR / "augmented_images"
augmented_images_dir.mkdir(exist_ok=True)

# Define transformations
no_transform = A.Compose([])
h_flip = A.Compose([A.HorizontalFlip(p=1.0)])
v_flip = A.Compose([A.VerticalFlip(p=1.0)])
hv_flip = A.Compose([A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0)])

transforms = [
    (no_transform, ""),
    (h_flip, "_hflip"),
    (v_flip, "_vflip"),
    (hv_flip, "_hvflip")
]

# Create list to store augmented data
augmented_data = []

# Process each image in train_cleaned
for row in tqdm(train_cleaned.iter_rows(named=True), total=len(train_cleaned)):
    img_path = DATA_DIR / row['image_path']
    image = cv2.imread(str(img_path))
    
    if image is None:
        continue
    
    # Get the base filename without extension
    base_name = row['image_path'].replace('train/', '').replace('.jpg', '')
    
    for transform, suffix in transforms:
        # Apply transformation
        augmented = transform(image=image)['image']
        
        # Save augmented image
        aug_filename = f"{base_name}{suffix}.jpg"
        aug_path = augmented_images_dir / aug_filename
        cv2.imwrite(str(aug_path), augmented)
        
        # Create new row with augmented image path
        new_row = {
            'image_path': f"augmented_images/{aug_filename}",
            'Dry_Clover_g': row['Dry_Clover_g'],
            'Dry_Dead_g': row['Dry_Dead_g'],
            'Dry_Green_g': row['Dry_Green_g'],
            'Dry_Total_g': row['Dry_Total_g'],
            'GDM_g': row['GDM_g']
        }
        augmented_data.append(new_row)

# Create new dataframe with augmented data
train_augmented = pl.DataFrame(augmented_data)

# Combine original and augmented data
train_cleaned_expanded = pl.concat([train_cleaned, train_augmented])

print(f"Original dataset size: {len(train_cleaned)}")
print(f"Expanded dataset size: {len(train_cleaned_expanded)}")

100%|██████████| 357/357 [00:50<00:00,  7.12it/s]

Original dataset size: 357
Expanded dataset size: 1785


image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,f64,f64,f64,f64,f64
"""train/ID1011485656.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1012260530.jpg""",0.0,0.0,7.6,7.6,7.6
"""train/ID1025234388.jpg""",6.05,0.0,0.0,6.05,6.05
"""train/ID1028611175.jpg""",0.0,30.9703,24.2376,55.2079,24.2376
"""train/ID1035947949.jpg""",0.4343,23.2239,10.5261,34.1844,10.9605
"""train/ID1036339023.jpg""",23.0755,2.6135,32.191,57.88,55.2665
"""train/ID1049634115.jpg""",1.5083,3.0167,13.575,18.1,15.0833
"""train/ID1051144034.jpg""",55.32,0.0,0.0,55.32,55.32
"""train/ID1052620238.jpg""",0.0,11.2291,20.1707,31.3998,20.1707


In [28]:
train_cleaned_expanded.sort("image_path")

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,f64,f64,f64,f64,f64
"""augmented_images/ID1011485656.…",0.0,31.9984,16.2751,48.2735,16.275
"""augmented_images/ID1011485656_…",0.0,31.9984,16.2751,48.2735,16.275
"""augmented_images/ID1011485656_…",0.0,31.9984,16.2751,48.2735,16.275
"""augmented_images/ID1011485656_…",0.0,31.9984,16.2751,48.2735,16.275
"""augmented_images/ID1012260530.…",0.0,0.0,7.6,7.6,7.6
…,…,…,…,…,…
"""train/ID975115267.jpg""",40.03,0.0,0.8,40.83,40.83
"""train/ID978026131.jpg""",24.6445,4.1948,12.0601,40.8994,36.7046
"""train/ID980538882.jpg""",0.0,1.1457,91.6543,92.8,91.6543


In [27]:
from cProfile import label
from albumentations import (Compose, RandomCrop, HorizontalFlip, VerticalFlip, Normalize)
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import cv2
import torch
from PIL import Image
import pandas as pd

class BiomassDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_dir: str, transform=None):
        self.df = df.reset_index(drop=True)
        # Ensure all label columns are float type
        label_cols = self.df.columns[1:]
        for col in label_cols:
            self.df[col] = pd.to_numeric(self.df[col], errors='coerce')
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.img_dir / self.df.iloc[idx]["image_path"]
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        
        labels = torch.tensor(self.df.iloc[idx][1:].values.astype('float32'), dtype=torch.float32)

        return image, labels
    
train_dataset = BiomassDataset(
    df=train_cleaned.to_pandas(),
    img_dir=DATA_DIR,
    transform=Compose([
        RandomCrop(width=256, height=256),
        HorizontalFlip(p=0.5),
        VerticalFlip(p=0.5),
        Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
for images, labels in train_loader:
    print(images.shape)
    print(labels.shape)
    break

torch.Size([16, 3, 256, 256])
torch.Size([16, 5])


In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class BiomassDataset(Dataset):
    def __init__(self, df, img_dir, mode='train'):
        self.df = df
        self.img_dir = img_dir
        self.mode = mode
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])  
        self.target_col = 'target' if mode in ['train', 'val'] else None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        image = Image.open(self.img_dir / img_path).convert('RGB')
        target_name = row['target_name']
        if self.transform:
            image = self.transform(image)
        if self.mode == 'test':
            return image, target_name
        
        label = torch.tensor(row['target'], dtype=torch.float32)
        return image, target_name, label

train_dataset = BiomassDataset(train_df, DATA_DIR, mode='train')
val_dataset = BiomassDataset(val_df, DATA_DIR, mode='val')
test_dataset = BiomassDataset(test_cleaned, DATA_DIR, mode='test')
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

for images, target_names, labels in train_loader:
    print(images.shape, target_names.shape, labels.shape)
    break

torch.Size([32, 3, 224, 224]) torch.Size([32]) torch.Size([32])


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Detect CUDA device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

class BiomassModel(nn.Module):
    def __init__(self, num_targets, num_target_names=5):
        super(BiomassModel, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        # Embedding for target_name
        self.target_name_embedding = nn.Embedding(num_target_names, 16)
        
        self.classifier = nn.Sequential(
            nn.Linear(32 * 56 * 56 + 16, 128),
            nn.ReLU(),
            nn.Linear(128, num_targets),
        )

    def forward(self, x, target_names):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        # Embed target_names
        target_name_emb = self.target_name_embedding(target_names)
        # Concatenate image features with target_name embedding
        x = torch.cat([x, target_name_emb], dim=1)
        x = self.classifier(x)
        return x
    
num_targets = 1
model = BiomassModel(num_targets, num_target_names=5)
model = model.to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, target_names, labels in train_loader:
        images = images.to(device)
        target_names = target_names.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images, target_names)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    epoch_loss = running_loss / len(train_dataset)
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')


Using device: cuda


In [7]:
outputs = []
model.eval()
with torch.no_grad():
    for images, target_names in test_loader:
        images = images.to(device)
        target_names = target_names.to(device)
        preds = model(images, target_names)
        preds = preds.cpu()
        outputs.append(preds)

outputs = torch.cat(outputs, dim=0).numpy()
outputs

array([[19.790606],
       [19.68659 ],
       [20.451479],
       [20.867886],
       [20.426811]], dtype=float32)

In [8]:
sample_submission = pl.read_csv(DATA_DIR / "sample_submission.csv")
# outputs is already a concatenated tensor from the previous cell
sample_submission = sample_submission.with_columns(pl.Series("target", outputs.flatten()))
sample_submission

sample_id,target
str,f32
"""ID1001187975__Dry_Clover_g""",19.790606
"""ID1001187975__Dry_Dead_g""",19.68659
"""ID1001187975__Dry_Green_g""",20.451479
"""ID1001187975__Dry_Total_g""",20.867886
"""ID1001187975__GDM_g""",20.426811
